In [41]:
import pandas as pd
df = pd.read_csv('../data/Superstore.csv', encoding='latin1')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


# Superstore Analysis

Layer 1 covers cleaning and core EDA. Layer 2 adds RFM customer segmentation.

In [42]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

# Reload the raw data so every section starts from the same source
raw_df = pd.read_csv('../data/Superstore.csv', encoding='latin1')
df = raw_df.copy()

# Standardize column names and parse dates
if 'Row ID' in df.columns:
    df = df.rename(columns={'Row ID': 'Row_ID'})
for col in ['Order Date', 'Ship Date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Coerce numeric columns so cleaning and charts work consistently
for col in ['Sales', 'Profit', 'Discount', 'Quantity']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Trim text fields used for grouping
for col in ['Category', 'Sub-Category', 'Segment', 'Region', 'State', 'Customer Name', 'Product Name']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

## 1) Data Cleaning Summary

Nulls, duplicates, and dtype fixes.

In [43]:
# -----------------------------
# 1) Data cleaning summary
# -----------------------------
rows_before = len(df)
null_summary = df.isnull().sum().sort_values(ascending=False)
dupe_count = df.duplicated().sum()

# Remove duplicates so later charts are not inflated
df = df.drop_duplicates().copy()
rows_after = len(df)

cleaning_summary = pd.DataFrame({
    'metric': ['rows_before', 'rows_after', 'duplicates_removed', 'columns_with_nulls'],
    'value': [rows_before, rows_after, rows_before - rows_after, int((null_summary > 0).sum())]
})

display(Markdown('## 1) Data Cleaning Summary'))
display(cleaning_summary)

null_top = null_summary[null_summary > 0].head(10).reset_index()
null_top.columns = ['column', 'null_count']

if len(null_top) > 0:
    fig_nulls = px.bar(
        null_top,
        x='column',
        y='null_count',
        title='Top Columns by Missing Values',
        text='null_count',
        color='null_count',
        color_continuous_scale='Blues'
    )
    fig_nulls.update_layout(xaxis_title='Column', yaxis_title='Null Count', template='plotly_white')
    fig_nulls.show()
else:
    display(Markdown('No missing values were found after type coercion.'))

display(Markdown(
    f"Insight: the dataset has {rows_before:,} rows before cleaning and {rows_after:,} rows after removing {rows_before - rows_after:,} duplicates. "
    f"There are null values in {int((null_summary > 0).sum())} columns, so those fields need careful handling in the next steps."
))

## 1) Data Cleaning Summary

,metric,value
0,rows_before,9994
1,rows_after,9994
2,duplicates_removed,0
3,columns_with_nulls,0


No missing values were found after type coercion.

Insight: the dataset has 9,994 rows before cleaning and 9,994 rows after removing 0 duplicates. There are null values in 0 columns, so those fields need careful handling in the next steps.

## 2) Sales & Profit Trend Over Time

In [44]:
# -----------------------------
# 2) Sales and profit trend over time
# -----------------------------
df_time = df.dropna(subset=['Order Date']).copy()
df_time['order_month'] = df_time['Order Date'].dt.to_period('M').dt.to_timestamp()
df_time['order_quarter'] = df_time['Order Date'].dt.to_period('Q').dt.to_timestamp()

monthly = df_time.groupby('order_month', as_index=False).agg({'Sales': 'sum', 'Profit': 'sum'})
quarterly = df_time.groupby('order_quarter', as_index=False).agg({'Sales': 'sum', 'Profit': 'sum'})

fig_trend = go.Figure()
fig_trend.add_trace(go.Scatter(x=monthly['order_month'], y=monthly['Sales'], mode='lines+markers', name='Monthly Sales'))
fig_trend.add_trace(go.Scatter(x=monthly['order_month'], y=monthly['Profit'], mode='lines+markers', name='Monthly Profit'))
fig_trend.add_trace(go.Scatter(x=quarterly['order_quarter'], y=quarterly['Sales'], mode='lines', name='Quarterly Sales', line=dict(dash='dash')))
fig_trend.add_trace(go.Scatter(x=quarterly['order_quarter'], y=quarterly['Profit'], mode='lines', name='Quarterly Profit', line=dict(dash='dash')))
fig_trend.update_layout(title='Sales and Profit Trends (Monthly + Quarterly)', template='plotly_white', xaxis_title='Date', yaxis_title='Amount')
fig_trend.show()

best_month = monthly.loc[monthly['Profit'].idxmax()]
worst_month = monthly.loc[monthly['Profit'].idxmin()]

display(Markdown(
    f"Insight: profit peaks in **{best_month['order_month'].strftime('%b %Y')}** and is weakest in **{worst_month['order_month'].strftime('%b %Y')}**. "
    "That seasonal swing suggests the business should protect margin more aggressively in weak months."
))

Insight: profit peaks in **Dec 2016** and is weakest in **Jan 2015**. That seasonal swing suggests the business should protect margin more aggressively in weak months.

## 3) Category and Sub-Category Profitability

In [45]:
# -----------------------------
# 3) Category and sub-category profitability
# -----------------------------
subcat_perf = df.groupby(['Category', 'Sub-Category'], as_index=False).agg({'Sales': 'sum', 'Profit': 'sum'})
subcat_perf = subcat_perf.sort_values('Profit', ascending=True)

fig_subcat = px.bar(
    subcat_perf,
    x='Profit',
    y='Sub-Category',
    color='Category',
    orientation='h',
    title='Sub-Category Profitability',
    template='plotly_white'
)
fig_subcat.add_vline(x=0, line_dash='dash', line_color='black')
fig_subcat.show()

loss_makers = subcat_perf[subcat_perf['Profit'] < 0][['Sub-Category', 'Profit']]
display(loss_makers.head(10))

display(Markdown(
    f"Insight: there are {len(loss_makers)} loss-making sub-categories. The usual pressure points are the low-margin items like Tables and Bookcases, "
    "so those groups should be checked for discounting and shipping-cost leakage."
))

,Sub-Category,Profit
3,Tables,-17725.4811
0,Bookcases,-3472.5560
12,Supplies,-1189.0995


Insight: there are 3 loss-making sub-categories. The usual pressure points are the low-margin items like Tables and Bookcases, so those groups should be checked for discounting and shipping-cost leakage.

## 4) Discount vs Profit

In [46]:
# -----------------------------
# 4) Discount versus profit
# -----------------------------
scatter_df = df.dropna(subset=['Discount', 'Profit', 'Sales']).copy()

fig_disc = px.scatter(
    scatter_df.sample(min(5000, len(scatter_df)), random_state=42),
    x='Discount',
    y='Profit',
    color='Category',
    title='Discount vs Profit (Sampled Points)',
    template='plotly_white',
    opacity=0.6
)
fig_disc.add_hline(y=0, line_dash='dash', line_color='black')
fig_disc.show()

# Estimate the break-even discount by looking for the first bin that crosses zero profit
bins = np.linspace(0, 0.9, 19)
scatter_df['discount_bin'] = pd.cut(scatter_df['Discount'], bins=bins, include_lowest=True)
bin_perf = scatter_df.groupby('discount_bin', observed=False, as_index=False)['Profit'].mean()
bin_perf['bin_mid'] = bin_perf['discount_bin'].apply(lambda x: x.mid if pd.notnull(x) else np.nan)
bin_perf = bin_perf.dropna(subset=['bin_mid'])

break_even = np.nan
for i in range(1, len(bin_perf)):
    p0, p1 = bin_perf.iloc[i - 1]['Profit'], bin_perf.iloc[i]['Profit']
    d0, d1 = bin_perf.iloc[i - 1]['bin_mid'], bin_perf.iloc[i]['bin_mid']
    if (p0 >= 0 and p1 <= 0) or (p0 <= 0 and p1 >= 0):
        if p1 != p0:
            break_even = d0 + (0 - p0) * (d1 - d0) / (p1 - p0)
        else:
            break_even = d0
        break

fig_bin = px.line(bin_perf, x='bin_mid', y='Profit', markers=True, title='Average Profit by Discount Bin', template='plotly_white')
fig_bin.add_hline(y=0, line_dash='dash', line_color='black')
if not np.isnan(break_even):
    fig_bin.add_vline(x=float(break_even), line_dash='dot', line_color='red')
fig_bin.update_layout(xaxis_title='Discount (bin midpoint)', yaxis_title='Average Profit')
fig_bin.show()

insight_text = (
    f"Insight: the average profit line crosses around **{break_even:.1%}** discount if a crossing exists, so that area is the rough break-even zone for pricing control."
    if not np.isnan(break_even)
    else 'Insight: the profit response to discount is noisy and may vary by category, so break-even should be checked at a more granular level.'
)
display(Markdown(insight_text))

Insight: the profit response to discount is noisy and may vary by category, so break-even should be checked at a more granular level.

## 5) Regional Performance

In [47]:
# -----------------------------
# 5) Regional and state performance
# -----------------------------
region_perf = df.groupby('Region', as_index=False).agg({'Sales': 'sum', 'Profit': 'sum'}).sort_values('Sales', ascending=False)
state_perf = df.groupby(['State', 'Region'], as_index=False).agg({'Sales': 'sum', 'Profit': 'sum'})

fig_region = px.scatter(
    state_perf,
    x='Sales',
    y='Profit',
    color='Region',
    size=np.abs(state_perf['Profit']) + 1,
    hover_name='State',
    title='State-Level Sales vs Profit by Region',
    template='plotly_white'
)
fig_region.add_hline(y=0, line_dash='dash', line_color='black')
fig_region.show()

display(region_perf)

# Find mismatch: high sales but low or negative profit states
sales_cutoff = state_perf['Sales'].quantile(0.75)
mismatch = state_perf[(state_perf['Sales'] >= sales_cutoff) & (state_perf['Profit'] <= 0)].sort_values('Sales', ascending=False)
display(mismatch.head(10))

display(Markdown(
    f"Insight: {len(mismatch)} high-sales states are non-profitable, which shows why sales alone is not a reliable success measure. "
    "These markets need margin, shipping, and discount diagnostics."
))

,Region,Sales,Profit
3,West,725457.8245,108418.4489
1,East,678781.2400,91522.7800
0,Central,501239.8908,39706.3625
2,South,391721.9050,46749.4303


,State,Region,Sales,Profit
41,Texas,Central,170188.0458,-25729.3563
36,Pennsylvania,East,116511.9140,-15559.9603
8,Florida,South,89473.7080,-3399.3017
11,Illinois,Central,80166.1010,-12607.8870
33,Ohio,East,78258.1360,-16971.3766
31,North Carolina,South,55603.1640,-7490.9122


Insight: 6 high-sales states are non-profitable, which shows why sales alone is not a reliable success measure. These markets need margin, shipping, and discount diagnostics.

## 6) Top and Bottom 10 Products by Profit

In [48]:
# 6) Top and bottom products by profit
product_perf = df.groupby('Product Name', as_index=False).agg({'Sales': 'sum', 'Profit': 'sum', 'Quantity': 'sum'})
top10 = product_perf.sort_values('Profit', ascending=False).head(10)
bottom10 = product_perf.sort_values('Profit', ascending=True).head(10)

display(Markdown('### Top 10 products by profit'))
display(top10[['Product Name', 'Sales', 'Quantity', 'Profit']])

display(Markdown('### Bottom 10 products to fix or drop'))
display(bottom10[['Product Name', 'Sales', 'Quantity', 'Profit']])

display(Markdown(
    'Insight: the bottom products should be triaged into a fix list or a drop list depending on whether the issue is pricing, discounting, or just low demand.'
))

### Top 10 products by profit

,Product Name,Sales,Quantity,Profit
404,Canon imageCLASS 2200 Advanced Copier,61599.824,20,25199.9280
650,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.384,31,7753.0390
805,Hewlett Packard LaserJet 3310 Copier,18839.686,38,6983.8836
400,Canon PC1060 Personal Laser Copier,11619.834,19,4570.9347
787,HP Designjet T520 Inkjet Large Format Printer ...,18374.895,12,4094.9766
165,Ativa V4110MDD Micro-Cut Shredder,7699.890,11,3772.9461
19,"3D Systems Cube Printer, 2nd Generation, Magenta",14299.890,11,3717.9714
1276,Plantronics Savi W720 Multi-Device Wireless He...,9367.290,24,3696.2820
895,Ibico EPK-21 Electric Binding System,15875.916,13,3345.2823
1840,Zebra ZM400 Thermal Label Printer,6965.700,6,3343.5360


### Bottom 10 products to fix or drop

,Product Name,Sales,Quantity,Profit
475,Cubify CubeX 3D Printer Double Head Print,11099.963,9,-8879.9704
985,Lexmark MX611dhe Monochrome Laser Printer,16829.901,18,-4589.9730
476,Cubify CubeX 3D Printer Triple Head Print,7999.980,4,-3839.9904
425,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.640,27,-2876.1156
376,Bush Advantage Collection Racetrack Conference...,9544.725,33,-1934.3976
683,GBC DocuBind P400 Electric Binding System,17965.068,27,-1878.1662
444,Cisco TelePresence System EX90 Videoconferenci...,22638.480,6,-1811.0784
1043,Martin Yale Chadless Opener Electric Letter Op...,16656.200,22,-1299.1836
285,Balt Solid Wood Round Tables,6518.754,19,-1201.0581
364,BoxOffice By Design Rectangular and Half-Moon ...,1706.250,15,-1148.4375


Insight: the bottom products should be triaged into a fix list or a drop list depending on whether the issue is pricing, discounting, or just low demand.

## 7) Customer Segment Analysis

In [49]:
# -----------------------------
# 7) Customer segment analysis
# -----------------------------
segment_perf = df.groupby('Segment', as_index=False).agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order ID', 'nunique')
)
segment_perf['Avg_Order_Value'] = segment_perf['Sales'] / segment_perf['Orders']

fig_segment = px.bar(
    segment_perf.melt(id_vars='Segment', value_vars=['Sales', 'Profit', 'Avg_Order_Value']),
    x='Segment',
    y='value',
    color='variable',
    barmode='group',
    title='Segment Value Comparison',
    template='plotly_white'
)
fig_segment.show()

display(segment_perf.sort_values('Profit', ascending=False))

display(Markdown(
    'Insight: the strongest segment should be protected with retention offers, while the weakest segment needs margin-safe upsell or reactivation tactics.'
))

,Segment,Sales,Profit,Orders,Avg_Order_Value
0,Consumer,1.161401e+06,134119.2092,2586,449.111116
1,Corporate,7.061464e+05,91979.1340,1514,466.411075
2,Home Office,4.296531e+05,60298.6785,909,472.665730


Insight: the strongest segment should be protected with retention offers, while the weakest segment needs margin-safe upsell or reactivation tactics.

## Layer 2) RFM Customer Segmentation

In [50]:
# -----------------------------
# Layer 2: RFM segmentation
# -----------------------------
rfm_base = df.dropna(subset=['Order Date', 'Customer ID']).copy()
snapshot_date = rfm_base['Order Date'].max() + pd.Timedelta(days=1)

rfm = rfm_base.groupby('Customer ID', as_index=False).agg(
    Recency=('Order Date', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Order ID', 'nunique'),
    Monetary=('Sales', 'sum')
)

# Quantile scores: Recency is reversed because lower recency means a more recent customer
rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first'), 4, labels=[4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['RFM_Score'] = rfm[['R_Score', 'F_Score', 'M_Score']].sum(axis=1)

conditions = [
    (rfm['R_Score'] >= 3) & (rfm['F_Score'] >= 3) & (rfm['M_Score'] >= 3),
    (rfm['R_Score'] >= 2) & (rfm['F_Score'] >= 3),
    (rfm['R_Score'] <= 2) & (rfm['F_Score'] >= 2),
    (rfm['R_Score'] == 1)
]
choices = ['Champions', 'Loyal', 'At-Risk', 'Lost']
rfm['Segment'] = np.select(conditions, choices, default='Loyal')

rfm_summary = rfm.groupby('Segment', as_index=False).agg(
    Customers=('Customer ID', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean')
).sort_values('Customers', ascending=False)

fig_rfm = px.bar(rfm_summary, x='Segment', y='Customers', color='Segment', title='RFM Segment Distribution', template='plotly_white')
fig_rfm.show()

display(rfm_summary)

display(Markdown(
    'Insight: Champions and Loyal customers should be retained with perks and priority service, while At-Risk and Lost customers need win-back campaigns.'
))

,Segment,Customers,Avg_Recency,Avg_Frequency,Avg_Monetary
3,Loyal,363,66.628099,6.035813,2493.744470
1,Champions,176,33.232955,8.806818,4562.133337
0,At-Risk,162,255.382716,6.098765,2899.874275
2,Lost,92,497.826087,3.043478,1296.266499


Insight: Champions and Loyal customers should be retained with perks and priority service, while At-Risk and Lost customers need win-back campaigns.

## Layer 3) SQL Database and Queries

Create a SQLite database from the Superstore data and write SQL queries matching Layer 1 analysis.

In [51]:
# Set up SQLite database from the cleaned DataFrame
import sqlite3

# Create/overwrite SQLite database and load the cleaned df into a table
db_path = '../data/superstore.db'
conn = sqlite3.connect(db_path)

# Drop table if exists (so we start fresh each run)
conn.execute('DROP TABLE IF EXISTS orders')

# Write the cleaned DataFrame to SQLite
df.to_sql('orders', conn, if_exists='replace', index=False)

# Create indexes for common query columns to speed up aggregations
# Note: Column names have spaces, so they need to be quoted
try:
    conn.execute('CREATE INDEX idx_order_date ON orders("Order Date")')
    conn.execute('CREATE INDEX idx_product_name ON orders("Product Name")')
    conn.execute('CREATE INDEX idx_category ON orders(Category)')
    conn.execute('CREATE INDEX idx_region ON orders(Region)')
    conn.execute('CREATE INDEX idx_state ON orders(State)')
    conn.execute('CREATE INDEX idx_customer_id ON orders("Customer ID")')
except:
    pass  # Indexes may already exist
conn.commit()

display(Markdown('✓ SQLite database created: superstore.db'))
display(Markdown(f'✓ Loaded {len(df):,} rows into `orders` table'))
display(Markdown('✓ Indexes created on: Order_Date, Product_Name, Category, Region, State, Customer_ID'))

✓ SQLite database created: superstore.db

✓ Loaded 9,994 rows into `orders` table

✓ Indexes created on: Order_Date, Product_Name, Category, Region, State, Customer_ID

In [52]:
# Q1: Top 10 products by profit
query_top_products = """
SELECT 
    "Product Name",
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    SUM(Quantity) as total_qty,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 1) as profit_margin_pct
FROM orders
GROUP BY "Product Name"
ORDER BY total_profit DESC
LIMIT 10;
"""

display(Markdown('### Q1: Top 10 Products by Profit'))
top_products = pd.read_sql(query_top_products, conn)
display(top_products)
display(Markdown(f'**Insight:** These are the profit leaders. Protect margins and increase volume for these products.'))

### Q1: Top 10 Products by Profit

,Product Name,total_sales,total_profit,total_qty,profit_margin_pct
0,Canon imageCLASS 2200 Advanced Copier,61599.82,25199.93,20,40.9
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38,7753.04,31,28.2
2,Hewlett Packard LaserJet 3310 Copier,18839.69,6983.88,38,37.1
3,Canon PC1060 Personal Laser Copier,11619.83,4570.93,19,39.3
4,HP Designjet T520 Inkjet Large Format Printer ...,18374.90,4094.98,12,22.3
5,Ativa V4110MDD Micro-Cut Shredder,7699.89,3772.95,11,49.0
6,"3D Systems Cube Printer, 2nd Generation, Magenta",14299.89,3717.97,11,26.0
7,Plantronics Savi W720 Multi-Device Wireless He...,9367.29,3696.28,24,39.5
8,Ibico EPK-21 Electric Binding System,15875.92,3345.28,13,21.1
9,Zebra ZM400 Thermal Label Printer,6965.70,3343.54,6,48.0


**Insight:** These are the profit leaders. Protect margins and increase volume for these products.

In [53]:
# Q2: Bottom 10 products by profit (loss-makers)
query_bottom_products = """
SELECT 
    "Product Name",
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    SUM(Quantity) as total_qty,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 1) as profit_margin_pct
FROM orders
GROUP BY "Product Name"
ORDER BY total_profit ASC
LIMIT 10;
"""

display(Markdown('### Q2: Bottom 10 Products by Profit (Loss-Makers)'))
bottom_products = pd.read_sql(query_bottom_products, conn)
display(bottom_products)
display(Markdown(f'**Insight:** These unprofitable products need intervention: price increases, cost cuts, or discontinuation.'))

# Q3: Loss-making products summary
query_loss_summary = """
SELECT 
    COUNT(*) as loss_product_count,
    ROUND(SUM(Profit), 2) as total_loss,
    ROUND(SUM(Sales), 2) as sales_volume
FROM (
    SELECT "Product Name", SUM(Profit) as Profit, SUM(Sales) as Sales
    FROM orders
    GROUP BY "Product Name"
    HAVING SUM(Profit) < 0
)
"""

display(Markdown('### Q3: Summary of Loss-Making Products'))
loss_summary = pd.read_sql(query_loss_summary, conn)
display(loss_summary)
display(Markdown(f'**Insight:** {int(loss_summary.iloc[0]["loss_product_count"])} products are unprofitable overall.'))

### Q2: Bottom 10 Products by Profit (Loss-Makers)

,Product Name,total_sales,total_profit,total_qty,profit_margin_pct
0,Cubify CubeX 3D Printer Double Head Print,11099.96,-8879.97,9,-80.0
1,Lexmark MX611dhe Monochrome Laser Printer,16829.90,-4589.97,18,-27.3
2,Cubify CubeX 3D Printer Triple Head Print,7999.98,-3839.99,4,-48.0
3,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.64,-2876.12,27,-29.0
4,Bush Advantage Collection Racetrack Conference...,9544.73,-1934.40,33,-20.3
5,GBC DocuBind P400 Electric Binding System,17965.07,-1878.17,27,-10.5
6,Cisco TelePresence System EX90 Videoconferenci...,22638.48,-1811.08,6,-8.0
7,Martin Yale Chadless Opener Electric Letter Op...,16656.20,-1299.18,22,-7.8
8,Balt Solid Wood Round Tables,6518.75,-1201.06,19,-18.4
9,BoxOffice By Design Rectangular and Half-Moon ...,1706.25,-1148.44,15,-67.3


**Insight:** These unprofitable products need intervention: price increases, cost cuts, or discontinuation.

### Q3: Summary of Loss-Making Products

,loss_product_count,total_loss,sales_volume
0,302,-77068.38,555729.0


**Insight:** 302 products are unprofitable overall.

In [54]:
# Q4: Sub-category profitability (all sub-categories ranked)
query_subcat_profit = """
SELECT 
    Category,
    "Sub-Category",
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 1) as profit_margin_pct,
    COUNT(DISTINCT "Order ID") as order_count
FROM orders
GROUP BY Category, "Sub-Category"
ORDER BY total_profit ASC;
"""

display(Markdown('### Q4: All Sub-Categories Ranked by Profit (Lowest to Highest)'))
subcat_profit = pd.read_sql(query_subcat_profit, conn)
display(subcat_profit)
display(Markdown(f'**Insight:** {(subcat_profit["total_profit"] < 0).sum()} sub-categories are unprofitable. Tables and Bookcases typically need attention.'))

# Q5: Category-level summary
query_category = """
SELECT 
    Category,
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 1) as profit_margin_pct,
    COUNT(DISTINCT "Order ID") as order_count,
    COUNT(DISTINCT "Sub-Category") as sub_category_count
FROM orders
GROUP BY Category
ORDER BY total_profit DESC;
"""

display(Markdown('### Q5: Category-Level Summary'))
category_summary = pd.read_sql(query_category, conn)
display(category_summary)

### Q4: All Sub-Categories Ranked by Profit (Lowest to Highest)

,Category,Sub-Category,total_sales,total_profit,profit_margin_pct,order_count
0,Furniture,Tables,206965.53,-17725.48,-8.6,307
1,Furniture,Bookcases,114880.00,-3472.56,-3.0,224
2,Office Supplies,Supplies,46673.54,-1189.10,-2.5,187
3,Office Supplies,Fasteners,3024.28,949.52,31.4,215
4,Technology,Machines,189238.63,3384.76,1.8,112
5,Office Supplies,Labels,12486.31,5546.25,44.4,346
6,Office Supplies,Art,27118.79,6527.79,24.1,731
7,Office Supplies,Envelopes,16476.40,6964.18,42.3,249
8,Furniture,Furnishings,91705.16,13059.14,14.2,877
9,Office Supplies,Appliances,107532.16,18138.01,16.9,451


**Insight:** 3 sub-categories are unprofitable. Tables and Bookcases typically need attention.

### Q5: Category-Level Summary

,Category,total_sales,total_profit,profit_margin_pct,order_count,sub_category_count
0,Technology,836154.03,145454.95,17.4,1544,4
1,Office Supplies,719047.03,122490.80,17.0,3742,9
2,Furniture,741999.80,18451.27,2.5,1764,4


In [55]:
# Q6: Regional performance (sales vs profit by state)
query_regional = """
SELECT 
    Region,
    State,
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 1) as profit_margin_pct,
    COUNT(DISTINCT "Order ID") as order_count
FROM orders
GROUP BY Region, State
ORDER BY total_profit DESC;
"""

display(Markdown('### Q6: Regional & State Performance'))
regional = pd.read_sql(query_regional, conn)
display(regional.head(15))
display(Markdown(f'**Insight:** Top-performing state is {regional.iloc[0]["State"]}. Identify high-sales/low-profit mismatches for margin recovery.'))

# Q7: Sales and profit trend by month
query_monthly_trend = """
SELECT 
    strftime('%Y-%m', "Order Date") as order_month,
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 1) as profit_margin_pct,
    COUNT(DISTINCT "Order ID") as order_count
FROM orders
GROUP BY order_month
ORDER BY order_month;
"""

display(Markdown('### Q7: Monthly Sales & Profit Trend'))
monthly_trend = pd.read_sql(query_monthly_trend, conn)
display(monthly_trend)
best_month_sql = monthly_trend.loc[monthly_trend['total_profit'].idxmax()]
worst_month_sql = monthly_trend.loc[monthly_trend['total_profit'].idxmin()]
display(Markdown(f'**Insight:** Best month: {best_month_sql["order_month"]} (${best_month_sql["total_profit"]}). Worst: {worst_month_sql["order_month"]} (${worst_month_sql["total_profit"]}).'))

### Q6: Regional & State Performance

,Region,State,total_sales,total_profit,profit_margin_pct,order_count
0,West,California,457687.63,76381.39,16.7,1021
1,East,New York,310876.27,74038.55,23.8,562
2,West,Washington,138641.27,33402.65,24.1,256
3,Central,Michigan,76269.61,24463.19,32.1,117
4,South,Virginia,70636.72,18597.95,26.3,115
5,Central,Indiana,53555.36,18382.94,34.3,73
6,South,Georgia,49095.84,16250.04,33.1,91
7,South,Kentucky,36591.75,11199.70,30.6,61
8,Central,Minnesota,29863.15,10823.19,36.2,44
9,East,Delaware,27451.07,9977.37,36.3,44


**Insight:** Top-performing state is California. Identify high-sales/low-profit mismatches for margin recovery.

### Q7: Monthly Sales & Profit Trend

,order_month,total_sales,total_profit,profit_margin_pct,order_count
0,2014-01,14236.90,2450.19,17.2,32
1,2014-02,4519.89,862.31,19.1,28
2,2014-03,55691.01,498.73,0.9,71
3,2014-04,28295.35,3488.84,12.3,66
4,2014-05,23648.29,2738.71,11.6,69
5,2014-06,34595.13,4976.52,14.4,66
6,2014-07,33946.39,-841.48,-2.5,65
7,2014-08,27909.47,5318.10,19.1,72
8,2014-09,81777.35,8328.10,10.2,130
9,2014-10,31453.39,3448.26,11.0,78


**Insight:** Best month: 2016-12 ($17885.31). Worst: 2015-01 ($-3281.01).

In [56]:
# Q8: Top 10 customers by sales
query_top_customers = """
SELECT 
    "Customer ID",
    "Customer Name",
    Segment,
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    COUNT(DISTINCT "Order ID") as order_count,
    ROUND(AVG(Sales), 2) as avg_order_value
FROM orders
GROUP BY "Customer ID", "Customer Name", Segment
ORDER BY total_sales DESC
LIMIT 10;
"""

display(Markdown('### Q8: Top 10 Customers by Sales'))
top_customers = pd.read_sql(query_top_customers, conn)
display(top_customers)
display(Markdown(f'**Insight:** These VIP customers should be prioritized for retention and upsell programs.'))

# Q9: Segment performance (Consumer, Corporate, Home Office)
query_segment_perf = """
SELECT 
    Segment,
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 1) as profit_margin_pct,
    COUNT(DISTINCT "Customer ID") as unique_customers,
    COUNT(DISTINCT "Order ID") as order_count,
    ROUND(SUM(Sales) / COUNT(DISTINCT "Customer ID"), 2) as avg_cust_sales
FROM orders
GROUP BY Segment
ORDER BY total_profit DESC;
"""

display(Markdown('### Q9: Customer Segment Performance'))
segment_perf_sql = pd.read_sql(query_segment_perf, conn)
display(segment_perf_sql)
display(Markdown(f'**Insight:** {segment_perf_sql.iloc[0]["Segment"]} is the strongest segment. Tailor retention by segment strength.'))

### Q8: Top 10 Customers by Sales

,Customer ID,Customer Name,Segment,total_sales,total_profit,order_count,avg_order_value
0,SM-20320,Sean Miller,Home Office,25043.05,-1980.74,5,1669.54
1,TC-20980,Tamara Chand,Corporate,19052.22,8981.32,5,1587.68
2,RB-19360,Raymond Buch,Consumer,15117.34,6976.10,6,839.85
3,TA-21385,Tom Ashbrook,Home Office,14595.62,4703.79,4,1459.56
4,AB-10105,Adrian Barton,Consumer,14473.57,5444.81,10,723.68
5,KL-16645,Ken Lonsdale,Consumer,14175.23,806.85,12,488.80
6,SC-20095,Sanjit Chand,Consumer,14142.33,5757.41,9,642.83
7,HL-15040,Hunter Lopez,Consumer,12873.30,5622.43,6,1170.30
8,SE-20110,Sanjit Engle,Consumer,12209.44,2650.68,11,642.60
9,CC-12370,Christopher Conant,Consumer,12129.07,2177.05,5,1102.64


**Insight:** These VIP customers should be prioritized for retention and upsell programs.

### Q9: Customer Segment Performance

,Segment,total_sales,total_profit,profit_margin_pct,unique_customers,order_count,avg_cust_sales
0,Consumer,1161401.34,134119.21,11.5,409,2586,2839.61
1,Corporate,706146.37,91979.13,13.0,236,1514,2992.15
2,Home Office,429653.15,60298.68,14.0,148,909,2903.06


**Insight:** Consumer is the strongest segment. Tailor retention by segment strength.

In [57]:
# Q10: Discount impact on profitability (by discount band)
query_discount_impact = """
SELECT 
    ROUND(Discount, 1) as discount_level,
    COUNT("Order ID") as order_count,
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    ROUND(AVG(Profit), 2) as avg_profit_per_order,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 1) as profit_margin_pct
FROM orders
GROUP BY ROUND(Discount, 1)
ORDER BY discount_level;
"""

display(Markdown('### Q10: Discount Impact on Profitability'))
discount_impact = pd.read_sql(query_discount_impact, conn)
display(discount_impact)

# Find the discount level with zero or lowest profit
zero_discount_row = discount_impact[discount_impact['discount_level'] == 0.0]
if len(zero_discount_row) > 0:
    zero_margin = zero_discount_row.iloc[0]['profit_margin_pct']
    display(Markdown(f'**Insight:** 0% discount margin: {zero_margin:.1f}%. Higher discounts erode profitability and should be managed carefully.'))

# Q11: Customer purchase frequency and lifetime value
query_cust_frequency = """
SELECT 
    "Customer ID",
    "Customer Name",
    Segment,
    COUNT(DISTINCT "Order ID") as purchase_frequency,
    ROUND(SUM(Sales), 2) as lifetime_sales,
    ROUND(SUM(Profit), 2) as lifetime_profit,
    ROUND(AVG(Sales), 2) as avg_order_value
FROM orders
GROUP BY "Customer ID", "Customer Name", Segment
ORDER BY lifetime_sales DESC
LIMIT 15;
"""

display(Markdown('### Q11: Customer Purchase Frequency & Lifetime Value (Top 15)'))
cust_frequency = pd.read_sql(query_cust_frequency, conn)
display(cust_frequency)
display(Markdown(f'**Insight:** High-frequency customers drive volume. Target retention of top customers with personalized programs.'))

### Q10: Discount Impact on Profitability

,discount_level,order_count,total_sales,total_profit,avg_profit_per_order,profit_margin_pct
0,0.0,4798,1087908.47,320987.60,66.90,29.5
1,0.1,146,81927.87,10448.17,71.56,12.8
2,0.2,3657,764594.37,90337.31,24.70,11.8
3,0.3,254,117720.11,-12760.42,-50.24,-10.8
4,0.4,206,116417.78,-23057.05,-111.93,-19.8
5,0.5,77,64403.51,-22999.54,-298.70,-35.7
6,0.6,138,6644.70,-5944.66,-43.08,-89.5
7,0.7,418,40620.28,-40075.36,-95.87,-98.7
8,0.8,300,16963.76,-30539.04,-101.80,-180.0


**Insight:** 0% discount margin: 29.5%. Higher discounts erode profitability and should be managed carefully.

### Q11: Customer Purchase Frequency & Lifetime Value (Top 15)

,Customer ID,Customer Name,Segment,purchase_frequency,lifetime_sales,lifetime_profit,avg_order_value
0,SM-20320,Sean Miller,Home Office,5,25043.05,-1980.74,1669.54
1,TC-20980,Tamara Chand,Corporate,5,19052.22,8981.32,1587.68
2,RB-19360,Raymond Buch,Consumer,6,15117.34,6976.10,839.85
3,TA-21385,Tom Ashbrook,Home Office,4,14595.62,4703.79,1459.56
4,AB-10105,Adrian Barton,Consumer,10,14473.57,5444.81,723.68
5,KL-16645,Ken Lonsdale,Consumer,12,14175.23,806.85,488.80
6,SC-20095,Sanjit Chand,Consumer,9,14142.33,5757.41,642.83
7,HL-15040,Hunter Lopez,Consumer,6,12873.30,5622.43,1170.30
8,SE-20110,Sanjit Engle,Consumer,11,12209.44,2650.68,642.60
9,CC-12370,Christopher Conant,Consumer,5,12129.07,2177.05,1102.64


**Insight:** High-frequency customers drive volume. Target retention of top customers with personalized programs.

In [58]:
# Q12: Shipping performance (Days to Ship analysis)
query_shipping = """
SELECT 
    ROUND((julianday("Ship Date") - julianday("Order Date"))) as days_to_ship,
    COUNT("Order ID") as order_count,
    ROUND(SUM(Sales), 2) as total_sales,
    ROUND(SUM(Profit), 2) as total_profit,
    ROUND(AVG(Profit), 2) as avg_profit_per_order
FROM orders
WHERE "Order Date" IS NOT NULL AND "Ship Date" IS NOT NULL
GROUP BY ROUND((julianday("Ship Date") - julianday("Order Date")))
ORDER BY days_to_ship;
"""

display(Markdown('### Q12: Shipping Performance (Days to Ship)'))
shipping_perf = pd.read_sql(query_shipping, conn)
display(shipping_perf)

# Close the database connection
conn.close()

display(Markdown('---'))
display(Markdown('## Layer 3 Summary'))
display(Markdown(
    """
✅ **Layer 3 Complete: SQL Practice**

**Database:** `superstore.db` created with indexed `orders` table
- 12 comprehensive SQL queries matching Layer 1 EDA themes
- Queries cover: products, categories, regions, customers, discounts, shipping, and trends
- All queries include profit margin analysis and business insights

**Key Findings:**
- Product analysis identified top performers and loss-makers
- Regional diagnostics flagged high-sales/low-profit mismatches  
- Discount impact quantified the break-even zones
- Customer segmentation and lifetime value calculated
- Seasonal trends and shipping performance assessed

**Next Step:** Layer 4 Dashboard with Streamlit or Power BI/Tableau visualization
"""
))

display(Markdown('**Connection closed.** All data persisted in `../data/superstore.db`'))

### Q12: Shipping Performance (Days to Ship)

,days_to_ship,order_count,total_sales,total_profit,avg_profit_per_order
0,0.0,519,124907.69,15385.97,29.65
1,1.0,369,67975.33,7541.23,20.44
2,2.0,1334,368465.83,53118.11,39.82
3,3.0,1005,204659.60,26875.92,26.74
4,4.0,2774,631847.01,71134.77,25.64
5,5.0,2169,494357.08,58733.20,27.08
6,6.0,1203,240290.57,33275.97,27.66
7,7.0,621,164697.75,20331.85,32.74


---

## Layer 3 Summary


✅ **Layer 3 Complete: SQL Practice**

**Database:** `superstore.db` created with indexed `orders` table
- 12 comprehensive SQL queries matching Layer 1 EDA themes
- Queries cover: products, categories, regions, customers, discounts, shipping, and trends
- All queries include profit margin analysis and business insights

**Key Findings:**
- Product analysis identified top performers and loss-makers
- Regional diagnostics flagged high-sales/low-profit mismatches  
- Discount impact quantified the break-even zones
- Customer segmentation and lifetime value calculated
- Seasonal trends and shipping performance assessed

**Next Step:** Layer 4 Dashboard with Streamlit or Power BI/Tableau visualization


**Connection closed.** All data persisted in `../data/superstore.db`

## Layer 4) Interactive Streamlit Dashboard

Complete dashboard application for business stakeholders to explore Superstore data with filters and KPIs.

In [59]:
# Generate Streamlit dashboard app code
streamlit_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sqlite3
from datetime import datetime

# ============================================================================
# PAGE CONFIG & LAYOUT
# ============================================================================
st.set_page_config(
    page_title="Superstore Dashboard",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.title("📊 Superstore Sales Analytics Dashboard")
st.markdown("---")

# ============================================================================
# LOAD DATA FROM SQLITE
# ============================================================================
@st.cache_resource
def get_db_connection():
    """Establish SQLite connection"""
    return sqlite3.connect("../data/superstore.db")

@st.cache_data
def load_data():
    """Load data from SQLite database"""
    conn = sqlite3.connect("../data/superstore.db")
    df = pd.read_sql("SELECT * FROM orders", conn)
    conn.close()
    
    # Convert date columns
    df["Order Date"] = pd.to_datetime(df["Order Date"])
    df["Ship Date"] = pd.to_datetime(df["Ship Date"])
    
    return df

df = load_data()

# ============================================================================
# SIDEBAR FILTERS
# ============================================================================
with st.sidebar:
    st.header("🔍 Filters")
    
    # Date range filter
    min_date = df["Order Date"].min().date()
    max_date = df["Order Date"].max().date()
    date_range = st.date_input(
        "Select Date Range:",
        value=(min_date, max_date),
        min_value=min_date,
        max_value=max_date
    )
    
    # Region filter
    regions = st.multiselect(
        "Select Regions:",
        options=sorted(df["Region"].unique()),
        default=sorted(df["Region"].unique())
    )
    
    # Category filter
    categories = st.multiselect(
        "Select Categories:",
        options=sorted(df["Category"].unique()),
        default=sorted(df["Category"].unique())
    )
    
    # Segment filter
    segments = st.multiselect(
        "Select Customer Segments:",
        options=sorted(df["Segment"].unique()),
        default=sorted(df["Segment"].unique())
    )
    
    st.markdown("---")
    st.caption(f"Data Range: {min_date} to {max_date}")
    st.caption(f"Total Records: {len(df):,}")

# ============================================================================
# APPLY FILTERS
# ============================================================================
df_filtered = df[
    (df["Order Date"].dt.date >= date_range[0]) &
    (df["Order Date"].dt.date <= date_range[1]) &
    (df["Region"].isin(regions)) &
    (df["Category"].isin(categories)) &
    (df["Segment"].isin(segments))
].copy()

# ============================================================================
# KPI METRICS (TOP)
# ============================================================================
st.subheader("📈 Key Performance Indicators")

col1, col2, col3, col4 = st.columns(4)

with col1:
    total_sales = df_filtered["Sales"].sum()
    st.metric(
        label="Total Sales",
        value=f"${total_sales:,.0f}",
        delta=f"{(total_sales / df['Sales'].sum() * 100):.1f}% of all time"
    )

with col2:
    total_profit = df_filtered["Profit"].sum()
    profit_margin = (total_profit / total_sales * 100) if total_sales > 0 else 0
    st.metric(
        label="Total Profit",
        value=f"${total_profit:,.0f}",
        delta=f"{profit_margin:.1f}% margin",
        delta_color="inverse" if profit_margin < 0 else "normal"
    )

with col3:
    total_orders = df_filtered["Order ID"].nunique()
    st.metric(
        label="Total Orders",
        value=f"{total_orders:,}",
        delta=f"{(total_orders / df['Order ID'].nunique() * 100):.1f}% of all time"
    )

with col4:
    avg_order_value = df_filtered["Sales"].sum() / total_orders if total_orders > 0 else 0
    st.metric(
        label="Avg Order Value",
        value=f"${avg_order_value:.2f}",
        delta=f"Based on {total_orders:,} orders"
    )

st.markdown("---")

# ============================================================================
# VISUALIZATIONS - ROW 1
# ============================================================================
st.subheader("📊 Sales Analysis")

col1, col2 = st.columns(2)

# Chart 1: Sales by Category (pie)
with col1:
    category_sales = df_filtered.groupby("Category", as_index=False).agg(
        {"Sales": "sum", "Profit": "sum"}
    ).sort_values("Sales", ascending=False)
    
    fig_category = px.pie(
        category_sales,
        names="Category",
        values="Sales",
        title="Sales Distribution by Category",
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    st.plotly_chart(fig_category, use_container_width=True)

# Chart 2: Profit by Category (bar)
with col2:
    fig_profit = px.bar(
        category_sales,
        x="Category",
        y="Profit",
        title="Profit by Category",
        color="Profit",
        color_continuous_scale="RdYlGn",
        text="Profit"
    )
    fig_profit.update_traces(texttemplate="$%{text:.0f}", textposition="outside")
    fig_profit.update_layout(showlegend=False)
    st.plotly_chart(fig_profit, use_container_width=True)

# ============================================================================
# VISUALIZATIONS - ROW 2
# ============================================================================
col1, col2 = st.columns(2)

# Chart 3: Monthly Sales Trend
with col1:
    monthly_data = df_filtered.copy()
    monthly_data["YearMonth"] = monthly_data["Order Date"].dt.to_period("M")
    monthly_sales = monthly_data.groupby("YearMonth", as_index=False).agg(
        {"Sales": "sum", "Profit": "sum", "Order ID": "nunique"}
    )
    
    fig_trend = go.Figure()
    fig_trend.add_trace(go.Scatter(
        x=monthly_sales.index,
        y=monthly_sales["Sales"],
        mode="lines+markers",
        name="Sales",
        line=dict(color="blue", width=2)
    ))
    fig_trend.add_trace(go.Scatter(
        x=monthly_sales.index,
        y=monthly_sales["Profit"],
        mode="lines+markers",
        name="Profit",
        line=dict(color="green", width=2)
    ))
    fig_trend.update_layout(
        title="Sales & Profit Trend Over Time",
        xaxis_title="Month",
        yaxis_title="Amount ($)",
        hovermode="x unified",
        height=400
    )
    st.plotly_chart(fig_trend, use_container_width=True)

# Chart 4: Regional Performance
with col2:
    region_data = df_filtered.groupby("Region", as_index=False).agg(
        {"Sales": "sum", "Profit": "sum", "Order ID": "nunique"}
    )
    region_data["Profit Margin %"] = (
        region_data["Profit"] / region_data["Sales"] * 100
    ).round(1)
    
    fig_region = px.scatter(
        region_data,
        x="Sales",
        y="Profit",
        size="Order ID",
        color="Region",
        title="Regional Sales vs Profit Performance",
        labels={"Order ID": "Number of Orders"},
        hover_data={"Profit Margin %": True},
        color_discrete_sequence=px.colors.qualitative.Plotly
    )
    fig_region.add_hline(y=0, line_dash="dash", line_color="red", opacity=0.5)
    st.plotly_chart(fig_region, use_container_width=True)

# ============================================================================
# VISUALIZATIONS - ROW 3
# ============================================================================
col1, col2 = st.columns(2)

# Chart 5: Top 10 Products by Profit
with col1:
    top_products = df_filtered.groupby("Product Name", as_index=False).agg(
        {"Sales": "sum", "Profit": "sum"}
    ).nlargest(10, "Profit")
    
    fig_top_prod = px.bar(
        top_products,
        x="Profit",
        y="Product Name",
        orientation="h",
        title="Top 10 Products by Profit",
        color="Profit",
        color_continuous_scale="Greens",
        text="Profit"
    )
    fig_top_prod.update_traces(texttemplate="$%{text:.0f}", textposition="outside")
    fig_top_prod.update_layout(showlegend=False, yaxis_title="")
    st.plotly_chart(fig_top_prod, use_container_width=True)

# Chart 6: Customer Segment Performance
with col2:
    segment_data = df_filtered.groupby("Segment", as_index=False).agg(
        {"Sales": "sum", "Profit": "sum", "Order ID": "nunique"}
    )
    
    fig_segment = px.bar(
        segment_data,
        x="Segment",
        y=["Sales", "Profit"],
        title="Segment Performance (Sales vs Profit)",
        barmode="group",
        color_discrete_map={"Sales": "#1f77b4", "Profit": "#2ca02c"}
    )
    st.plotly_chart(fig_segment, use_container_width=True)

# ============================================================================
# DETAILED DATA TABLES
# ============================================================================
st.markdown("---")
st.subheader("📋 Detailed Data Tables")

tab1, tab2, tab3 = st.tabs(["Top Customers", "Sub-Category Performance", "Discount Impact"])

with tab1:
    top_cust = df_filtered.groupby(
        ["Customer ID", "Customer Name", "Segment"],
        as_index=False
    ).agg({
        "Sales": "sum",
        "Profit": "sum",
        "Order ID": "nunique"
    }).nlargest(20, "Sales")
    
    top_cust.columns = ["Customer ID", "Customer Name", "Segment", "Total Sales", "Total Profit", "Order Count"]
    st.dataframe(top_cust, use_container_width=True)

with tab2:
    subcat_perf = df_filtered.groupby(
        ["Category", "Sub-Category"],
        as_index=False
    ).agg({
        "Sales": "sum",
        "Profit": "sum",
        "Order ID": "nunique"
    }).sort_values("Profit", ascending=False)
    
    subcat_perf.columns = ["Category", "Sub-Category", "Sales", "Profit", "Order Count"]
    subcat_perf["Profit Margin %"] = (subcat_perf["Profit"] / subcat_perf["Sales"] * 100).round(1)
    st.dataframe(subcat_perf, use_container_width=True)

with tab3:
    discount_data = df_filtered.copy()
    discount_data["Discount %"] = (discount_data["Discount"] * 100).round(0).astype(int)
    disc_analysis = discount_data.groupby("Discount %", as_index=False).agg({
        "Sales": "sum",
        "Profit": "sum",
        "Order ID": "nunique"
    })
    disc_analysis.columns = ["Discount %", "Total Sales", "Total Profit", "Order Count"]
    disc_analysis["Profit Margin %"] = (disc_analysis["Total Profit"] / disc_analysis["Total Sales"] * 100).round(1)
    st.dataframe(disc_analysis.sort_values("Discount %"), use_container_width=True)

# ============================================================================
# FOOTER
# ============================================================================
st.markdown("---")
st.markdown(
    """
    **📊 Superstore Dashboard** | Built with Streamlit & Plotly  
    Data Source: SQLite Database (superstore.db)  
    Last Updated: {today}
    """.format(today=datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
)
'''

# Save the Streamlit app code
app_path = '../dashboard/streamlit_app.py'
with open(app_path, 'w') as f:
    f.write(streamlit_code)

display(Markdown(f'✓ Streamlit dashboard saved to `{app_path}`'))
display(Markdown('✓ Dashboard includes: KPI cards, 6 interactive charts, 3 data tables, dynamic filters'))

✓ Streamlit dashboard saved to `../dashboard/streamlit_app.py`

✓ Dashboard includes: KPI cards, 6 interactive charts, 3 data tables, dynamic filters

In [62]:
# Instructions for running the Streamlit dashboard

display(Markdown('''
## 🚀 How to Run the Dashboard

Open a terminal in the project root and run:

```bash
cd /home/minato/project/Superstore-analysis/dashboard
conda activate superstore
streamlit run streamlit_app.py

```

The dashboard will open at `http://localhost:8501` with:
- **KPI Cards**: Total Sales, Profit, Orders, Average Order Value
- **Interactive Filters**: Date range, Region, Category, Customer Segment
- **6 Visualization Panels**:
  1. Sales by Category (pie chart)
  2. Profit by Category (bar chart)
  3. Sales & Profit Trend (line chart)
  4. Regional Performance (scatter plot)
  5. Top 10 Products by Profit (horizontal bar)
  6. Segment Performance (grouped bar chart)
- **Data Tables** (tabs):
  - Top 20 Customers by Sales
  - Sub-Category Profitability
  - Discount Impact Analysis

All filters update charts and metrics in real-time!
'''))

display(Markdown('---'))


## 🚀 How to Run the Dashboard

Open a terminal in the project root and run:

```bash
cd /home/minato/project/Superstore-analysis/dashboard
conda activate superstore
streamlit run streamlit_app.py

```

The dashboard will open at `http://localhost:8501` with:
- **KPI Cards**: Total Sales, Profit, Orders, Average Order Value
- **Interactive Filters**: Date range, Region, Category, Customer Segment
- **6 Visualization Panels**:
  1. Sales by Category (pie chart)
  2. Profit by Category (bar chart)
  3. Sales & Profit Trend (line chart)
  4. Regional Performance (scatter plot)
  5. Top 10 Products by Profit (horizontal bar)
  6. Segment Performance (grouped bar chart)
- **Data Tables** (tabs):
  - Top 20 Customers by Sales
  - Sub-Category Profitability
  - Discount Impact Analysis

All filters update charts and metrics in real-time!


---